# 第16章 事件驱动回测（akquant）

> 本 notebook 为《量化投资》配套代码，可在 Jupyter / Colab / Binder 中运行。

In [ ]:
# Colab 自举：在 Colab 上自动克隆仓库并安装 qfin（本地用 uv 运行会自动跳过）
import importlib.util, subprocess, sys, os
if importlib.util.find_spec('qfin') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git','clone','-q','https://github.com/albertandking/quant-investing.git'], check=False)
        os.chdir('quant-investing')
        subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'], check=False)
        subprocess.run([sys.executable,'scripts/make_sample_data.py'], check=False)


## 准备数据：取一只股票的 OHLCV（内置数据，断网可跑）

In [ ]:
from qfin import load_prices
df = (load_prices().query("symbol == 'STK001'")
      .sort_values('date').reset_index(drop=True))
print(df.head())

## akquant 事件驱动回测（需 `uv sync --extra quant`）
未安装时自动跳过。不同版本 API 可能略有差异，以 `help(akquant.Strategy)` 为准。

In [ ]:
try:
    import akquant as aq
    from akquant import Strategy

    class MaCross(Strategy):
        fast, slow = 5, 20
        def on_bar(self, bar):
            closes = self.get_history(self.slow, field='close')
            if closes is None or len(closes) < self.slow:
                return
            ma_fast = closes[-self.fast:].mean()
            ma_slow = closes.mean()
            pos = self.get_position(bar.symbol)
            if pos == 0 and ma_fast > ma_slow:
                self.buy(symbol=bar.symbol, quantity=100)
            elif pos > 0 and ma_fast < ma_slow:
                self.close_position(symbol=bar.symbol)

    result = aq.run_backtest(data=df, strategy=MaCross, symbols='STK001',
                             initial_cash=100_000.0, history_depth=30, t_plus_one=True)
    print(result)
    result.equity_curve.plot(title='akquant 事件驱动回测净值曲线')
except Exception as e:
    print('跳过 akquant 示例（未安装或 API 差异）：', e)
    print('安装：uv sync --extra quant')

## 不依赖框架的向量化对照（一定可跑）
用 shift(1) 保证 T+1 时序正确。

In [ ]:
from qfin import load_close, sma, sharpe_ratio, max_drawdown

px = load_close()['STK001']
sig = (sma(px, 5) > sma(px, 20)).astype(int)
ret = px.pct_change().fillna(0)
strat = sig.shift(1).fillna(0) * ret           # T+1 滞后
print('策略年化夏普：', round(sharpe_ratio(strat), 2))
print('最大回撤：', round(max_drawdown(strat), 3))